# Frontend에 Browser Live View 삽입하기

## 개요

Amazon DCV(NICE DCV)와 AWS Bedrock AgentCore Browser를 사용해 웹 애플리케이션에 실시간 브라우저 스트리밍과 사람의 브라우저 제어권 인수 기능을 통합하는 방법을 알아봅니다.

### 튜토리얼 세부 정보

| 항목                | 세부 정보                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 통합 가이드                                                                      |
| Agent 유형          | 단일                                                                             |
| Agentic Framework   | Nova Act                                                                         |
| LLM 모델            | Amazon Nova Act model                                                            |
| 튜토리얼 구성 요소  | DCV live streaming, AgentCore Browser, WebSocket 인증                            |
| 튜토리얼 분야       | 웹 개발                                                                          |
| 예제 난이도         | 중급                                                                             |
| 사용 SDK            | Amazon BedrockAgentCore Python SDK, Nova Act, Amazon DCV Web Client SDK          |

### 튜토리얼 아키텍처

이 튜토리얼에서는 Amazon DCV protocol과 AWS Bedrock AgentCore Browser를 사용해 브라우저 세션을 실시간으로 스트리밍하는 방법을 보여 줍니다.

### 튜토리얼 주요 기능

* Browser tool을 headless 방식으로 사용
* Nova Act와 Browser tool을 함께 사용
* React 애플리케이션에 Browser Live View 삽입

## 사전 요구 사항

이 튜토리얼을 실행하려면 다음이 필요합니다.

* Python 3.10+
* 구성된 AWS 자격 증명. IAM 역할/사용자에 다음 권한이 있어야 합니다. https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/browser-onboarding.html#browser-credentials-config
* Amazon Bedrock AgentCore SDK
* HTML/JavaScript 기본 지식

### 필수 Python package 설치

In [ ]:
# 필수 package 설치
!pip install bedrock-agentcore boto3 -q

## DCVjs SDK 다운로드 및 설정

DCVjs SDK를 사용하면 애플리케이션에서 브라우저를 실시간으로 스트리밍할 수 있습니다.

### 수행할 작업

1. AWS CloudFront에서 최신 DCVjs SDK **자동 다운로드**
2. 올바른 디렉터리 구조에 파일 **압축 해제 및 정리**
3. 모든 항목이 준비되었는지 **설치 검증**

SDK는 다음과 같이 구성됩니다.

```
dcv-sdk/
├── dcvjs-umd/          # ← 이 버전을 사용합니다.
│   ├── dcv.js
│   └── dcv/            # Video decoding용 worker file
├── dcvjs-esm/          # ES Module 버전
└── dcv-ui/             # 선택적 UI component
```

### 중요 참고 사항

- SDK는 static file로 제공해야 합니다(npm package로 import하지 않음).
- Video decoding을 위해 runtime에 worker file에 액세스할 수 있어야 합니다.
- 설치 상태는 자동으로 검증합니다.

**아래 셀을 실행해 SDK를 자동으로 다운로드하고 설정하세요.**

In [ ]:
import urllib.request
import zipfile
import os
import shutil
from pathlib import Path

# DCV SDK 구성
DCV_SDK_URL = "https://d1uj6qtbmh3dt5.cloudfront.net/webclientsdk/nice-dcv-web-client-sdk-1.9.100-952.zip"
DCV_SDK_DIR = "dcv-sdk"
ZIP_FILE = "dcv-sdk.zip"


def download_dcv_sdk():
    """CloudFront에서 DCV SDK를 다운로드합니다."""
    print("📦 Downloading DCV SDK...")
    print(f"   Source: {DCV_SDK_URL}")

    # 진행률을 표시하며 다운로드
    def report_progress(block_num, block_size, total_size):
        downloaded = block_num * block_size
        percent = min(downloaded * 100 / total_size, 100)
        print(f"\r   Progress: {percent:.1f}%", end="", flush=True)

    urllib.request.urlretrieve(DCV_SDK_URL, ZIP_FILE, report_progress)
    print()  # 진행률 표시 후 줄바꿈
    print("✅ Download complete!")


def extract_dcv_sdk():
    """DCV SDK를 dcv-sdk 디렉터리에 압축 해제합니다."""
    print("\n📂 Extracting DCV SDK...")

    # 기존 디렉터리가 있으면 삭제
    if os.path.exists(DCV_SDK_DIR):
        shutil.rmtree(DCV_SDK_DIR)
        print("   Removed old dcv-sdk directory")

    # 임시 압축 해제 디렉터리 생성
    temp_dir = "dcv-sdk-temp"
    if os.path.exists(temp_dir):
        shutil.rmtree(temp_dir)

    # Zip file을 임시 디렉터리에 압축 해제
    with zipfile.ZipFile(ZIP_FILE, "r") as zip_ref:
        zip_ref.extractall(temp_dir)

    print("   Extracted to temporary directory")

    # 중첩된 실제 SDK 디렉터리 찾기
    # 구조: dcv-sdk-temp/nice-dcv-web-client-sdk/[실제 file]
    nested_dir = os.path.join(temp_dir, "nice-dcv-web-client-sdk")

    if os.path.exists(nested_dir):
        # 중첩된 디렉터리를 대상 위치로 이동
        shutil.move(nested_dir, DCV_SDK_DIR)
        print("   Moved SDK files to dcv-sdk/")
    else:
        # 구조가 다르면 임시 디렉터리를 대상 이름으로 변경
        shutil.move(temp_dir, DCV_SDK_DIR)
        print("   Organized SDK files")

    # 정리
    if os.path.exists(temp_dir):
        shutil.rmtree(temp_dir)

    os.remove(ZIP_FILE)
    print("✅ Extraction complete!")
    print("   Cleaned up temporary files")


def verify_installation():
    """필요한 파일이 모두 있는지 확인합니다."""
    print("\n🔍 Verifying installation...")

    required_files = [
        "dcv-sdk/dcvjs-umd/dcv.js",
        "dcv-sdk/dcvjs-umd/dcv",  # 디렉터리
        "dcv-sdk/dcvjs-esm/dcv.js",
    ]

    all_good = True
    for file_path in required_files:
        if os.path.exists(file_path):
            file_type = "📁" if os.path.isdir(file_path) else "📄"
            print(f"   {file_type} {file_path} ✓")
        else:
            print(f"   ❌ {file_path} - NOT FOUND")
            all_good = False

    if all_good:
        print("\n✅ All required files are in place!")

        # Worker file 수 확인
        worker_dir = Path("dcv-sdk/dcvjs-umd/dcv")
        if worker_dir.exists():
            worker_files = list(worker_dir.glob("*.js")) + list(worker_dir.glob("*.wasm"))
            print(f"   Found {len(worker_files)} worker files for video decoding")

        # File 크기 표시
        dcv_js = Path("dcv-sdk/dcvjs-umd/dcv.js")
        if dcv_js.exists():
            size_mb = dcv_js.stat().st_size / (1024 * 1024)
            print(f"   Main SDK file size: {size_mb:.2f} MB")

        return True
    else:
        print("\n❌ Some files are missing. Please check the extraction.")

        # 실제로 존재하는 항목 표시
        print("\n🔍 Debugging - Current structure:")
        if os.path.exists(DCV_SDK_DIR):
            for root, dirs, files in os.walk(DCV_SDK_DIR):
                level = root.replace(DCV_SDK_DIR, "").count(os.sep)
                indent = " " * 2 * level
                print(f"{indent}{os.path.basename(root)}/")
                sub_indent = " " * 2 * (level + 1)
                for file in files[:5]:  # 처음 5개 file 표시
                    print(f"{sub_indent}{file}")
                if len(files) > 5:
                    print(f"{sub_indent}... and {len(files) - 5} more files")
                if level > 2:  # 탐색 깊이 제한
                    break
        return False


def show_directory_structure():
    """디렉터리 구조를 표시합니다."""
    print("\n📋 Directory Structure:")
    print("=" * 60)

    def print_tree(directory, prefix="", max_depth=3, current_depth=0):
        if current_depth >= max_depth:
            return

        try:
            contents = sorted(Path(directory).iterdir(), key=lambda x: (not x.is_dir(), x.name))
            for i, path in enumerate(contents):
                is_last = i == len(contents) - 1
                current_prefix = "└── " if is_last else "├── "

                # 주요 file 크기 표시
                size_info = ""
                if path.is_file() and path.suffix in [".js", ".wasm"]:
                    size_mb = path.stat().st_size / (1024 * 1024)
                    if size_mb > 0.1:
                        size_info = f" ({size_mb:.1f}MB)"

                print(f"{prefix}{current_prefix}{path.name}{'/' if path.is_dir() else ''}{size_info}")

                if path.is_dir() and current_depth < max_depth - 1:
                    extension = "    " if is_last else "│   "
                    print_tree(path, prefix + extension, max_depth, current_depth + 1)
        except PermissionError:
            pass

    print_tree(DCV_SDK_DIR)
    print("=" * 60)


# Main 실행
try:
    # 이미 설치되어 있는지 확인
    if os.path.exists("dcv-sdk/dcvjs-umd/dcv.js"):
        print("⚠️  DCV SDK already exists!")
        print("   Current installation is valid.")

        # 빠른 검증
        if verify_installation():
            show_directory_structure()
            print("\n✅ Existing DCV SDK is ready to use!")
            print("\nℹ️  To re-download, delete the 'dcv-sdk' directory first.")
    else:
        # 다운로드 및 설정
        download_dcv_sdk()
        extract_dcv_sdk()

    if verify_installation():
        show_directory_structure()
        print("\n🎉 DCV SDK is ready to use!")
        print("\nℹ️  The SDK is now available at: ./dcv-sdk/")
        print("   - UMD version: dcv-sdk/dcvjs-umd/dcv.js")
        print("   - ESM version: dcv-sdk/dcvjs-esm/dcv.js")
        print("   - UI components: dcv-sdk/dcv-ui/")
    else:
        print("\n⚠️  Installation incomplete. Please check for errors above.")

except KeyboardInterrupt:
    print("\n\n⚠️  Download cancelled by user")
except Exception as e:
    print(f"\n❌ Error during installation: {e}")
    import traceback

    traceback.print_exc()

## 아키텍처 흐름

DCVjs client SDK가 브라우저 실시간 스트리밍을 지원하는 방식은 다음과 같습니다.

```
┌──────────────────────┐
│   Python Backend     │
│ (AgentCore Browser)  │
└──────────┬───────────┘
           │
           │ 1. 브라우저 세션 생성
           │ 2. presigned URL 생성
           ↓
┌──────────────────────┐
│  DCV Streaming       │
│  Server              │
└──────────┬───────────┘
           │
           │ 3. DCV protocol로 스트리밍
           ↓
┌──────────────────────┐
│   Frontend           │
│ (DCVjs SDK)          │
│ - 인증               │
│ - Stream 렌더링      │
└──────────────────────┘
```

**처리 단계:**

1. AgentCore Browser Tool(Backend)이 브라우저 세션을 생성합니다.
2. DCV Streaming Server가 streaming URL과 auth token을 생성합니다.
3. 사람의 제어권 인수를 처리할 API server를 생성합니다. 제어권 인수에는 update_browser_stream API(https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/browser-update-stream.html)를 사용합니다.
4. DCVjs SDK(Frontend)가 연결해 stream을 렌더링합니다.
5. Frontend가 사람의 제어권 인수를 구현합니다. Agent 대신 사람이 브라우저 작업을 처리해야 할 때 유용하며, 작업 후 Agent에 제어권을 돌려줄 수 있습니다.
6. HTML에 실시간 브라우저 화면이 표시됩니다.

## 0단계: 환경 설정

먼저 필수 라이브러리를 import하고 AWS 구성을 설정합니다.

In [ ]:
from bedrock_agentcore.tools.browser_client import BrowserClient
import boto3

# boto3 session에서 AWS 리전 가져오기
session = boto3.Session()
region = session.region_name

print(f"Using AWS Region: {region}")
print("Environment setup complete!")

## 1단계: Browser Control API Server 생성

Human Take Over 기능을 지원하려면 브라우저 제어 API 호출을 처리하는 간단한 API server가 필요합니다. 이 server는 자동화 모드를 활성화하고 비활성화하는 endpoint를 제공합니다.

FastAPI를 사용해 이 API server를 구현해 보겠습니다.

In [ ]:
from fastapi import FastAPI
from fastapi.responses import JSONResponse
from fastapi.middleware.cors import CORSMiddleware  # 이 import 추가
from pydantic import BaseModel
import uvicorn
import threading


class AutomationStreamUpdate(BaseModel):
    streamStatus: str  # 'enabled' 또는 'disabled'


class BrowserStreamUpdateRequest(BaseModel):
    automationStreamUpdate: AutomationStreamUpdate


class BrowserControlServer:
    """브라우저 제어 API 엔드포인트를 처리하는 서버입니다."""

    def __init__(self, browser_client, port=8081):
        """제어 서버를 초기화합니다."""
        self.browser_client = browser_client
        self.port = port
        self.app = FastAPI(title="Browser Control API")
        self.server_thread = None
        self.is_running = False

        # Cross-origin request를 허용하도록 CORS middleware 추가
        self.app.add_middleware(
            CORSMiddleware,
            allow_origins=["*"],  # Demo를 위해 모든 origin 허용
            allow_credentials=True,
            allow_methods=["*"],  # 모든 method 허용
            allow_headers=["*"],  # 모든 header 허용
        )

        # Route 설정
        self._setup_routes()

    def _setup_routes(self):
        """FastAPI 라우트를 설정합니다."""

        @self.app.get("/")
        async def root():
            """루트 엔드포인트입니다."""
            return {"message": "Browser Control API is running."}

        @self.app.post("/api/update_browser_stream")
        async def update_browser_stream(request: BrowserStreamUpdateRequest):
            """브라우저 스트림 자동화 상태를 업데이트합니다."""
            try:
                stream_status = request.automationStreamUpdate.streamStatus
                print(f"Updating browser stream status to: {stream_status}")

                if stream_status == "disabled":
                    # 제어권 인수(자동화 비활성화)
                    self.browser_client.take_control()
                    print("✅ Human control enabled (automation disabled)")
                    return {"status": "success", "message": "Human control enabled"}

                elif stream_status == "enabled":
                    # 제어권 반환(자동화 활성화)
                    self.browser_client.release_control()
                    print("✅ Automation enabled")
                    return {"status": "success", "message": "Automation enabled"}

                else:
                    print(f"❌ Invalid stream status: {stream_status}")
                    return JSONResponse(
                        status_code=400,
                        content={
                            "status": "error",
                            "message": f"Invalid stream status: {stream_status}. Must be 'enabled' or 'disabled'",
                        },
                    )

            except Exception as e:
                print(f"❌ Error updating browser stream: {e}")
                return JSONResponse(status_code=500, content={"status": "error", "message": str(e)})

    def start(self):
        """API 서버를 시작합니다."""

        # Thread에서 server 실행
        def run_server():
            uvicorn.run(self.app, host="0.0.0.0", port=self.port, log_level="error")

        self.server_thread = threading.Thread(target=run_server, daemon=True)
        self.server_thread.start()
        self.is_running = True

        print(f"🌐 Browser Control API server running at: http://localhost:{self.port}")
        print("🔓 CORS enabled: Allowing requests from all origins")
        return f"http://localhost:{self.port}"


# 참고: Control server는 browser_client가 생성된 후 1단계에서 초기화됨
print("✅ Browser Control Server class defined")
print("The server will be initialized after the browser client is created.")

## 2단계: 브라우저 세션 생성 및 Streaming URL 가져오기

AgentCore Browser client를 초기화하고 Frontend streaming에 필요한 URL을 생성합니다.

In [ ]:
# BrowserClient 생성 및 세션 시작
browser_client = BrowserClient(region)
browser_client.start()

print("✅ Browser session started")

### 2.1단계: Streaming URL 가져오기

아래 코드를 실행해 실시간 스트리밍용 presigned URL을 가져옵니다.

In [ ]:
# Agent 연결에 사용할 automation endpoint WebSocket URL과 signed header 가져오기
ws_url, headers = browser_client.generate_ws_headers()
print(f"\n📡 WebSocket URL: {ws_url[:60]}...")
print(f"🔐 Headers configured: {list(headers.keys())}")

# 5분 후 만료되는 presigned Live View URL 가져오기
live_view_url = browser_client.generate_live_view_url(expires=300)
print("\n🔗 Live View URL generated (expires in 300 seconds)")
print(f"   URL: {live_view_url[:80]}...")

# Browser control server 생성 및 시작
control_server = BrowserControlServer(browser_client, port=8081)
control_api_url = control_server.start()
print(f"\n🌐 Control API server running at: {control_api_url}")
print(f"   API endpoint: {control_api_url}/api/update_browser_stream")

# Frontend에서 사용할 정보 저장
session_data = {
    "presignedUrl": live_view_url,
    "sessionId": "demo-session",
    "authToken": "demo-session",
}

print("\n✅ Ready to connect from frontend!")

## 3단계: Live View 생성 및 실행

이 단계에서는 다음 작업을 자동으로 수행합니다.

✅ DCV가 통합된 HTML 페이지 생성

✅ Presigned URL 자동 삽입

✅ 브라우저 제어를 위한 Human Take Over 버튼 추가

✅ 해상도 선택 control 추가(720p, 900p, 1080p, 1440p)

✅ 모든 필수 파일이 올바른 위치에 있는지 검증

✅ 로컬 web server 시작(port 8080)

✅ 브라우저 자동 열기

### 표시되는 내용

1. 실제 presigned URL이 포함된 HTML file 생성
2. File 구조 검증
3. Web server 시작
4. 브라우저가 열리고 실시간 브라우저 stream 표시

### Viewer 기능

- DCV를 통한 실시간 브라우저 스트리밍
- 브라우저를 수동으로 제어하는 Human Take Over 버튼
- 화면 크기를 조정하는 해상도 control(720p~1440p)
- 상태 표시기(Loading → Authenticating → Connected)
- 자동 오류 처리
- 정보 panel이 포함된 깔끔한 UI

**참고:** Web server는 계속 실행됩니다. 중지하려면 정지 버튼(■)을 클릭하거나 kernel을 다시 시작하세요.

In [ ]:
import http.server
import socketserver
import webbrowser
import time
import os

# 구성
HTML_FILE = "browser_live_view.html"
SERVER_PORT = 8080
SERVER_URL = f"http://localhost:{SERVER_PORT}/{HTML_FILE}"
CONTROL_API_URL = "http://localhost:8081"  # Control API server URL 설정


def create_html_with_url(presigned_url, session_id="demo-session"):
    """미리 서명된 URL이 삽입된 HTML 파일을 생성합니다."""
    print("📝 Creating HTML file with live view integration...")

    html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Browser Live View - AWS AgentCore</title>
    <style>
        * {{
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }}
        body {{
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            min-height: 100vh;
            padding: 20px;
        }}
        .container {{
            max-width: 1600px;
            margin: 0 auto;
        }}
        .header {{
            background: white;
            padding: 20px 30px;
            border-radius: 10px;
            margin-bottom: 20px;
            box-shadow: 0 4px 6px rgba(0,0,0,0.1);
        }}
        h1 {{
            color: #333;
            font-size: 28px;
            margin-bottom: 10px;
        }}
        .subtitle {{
            color: #666;
            font-size: 14px;
        }}
        .status {{
            padding: 15px 20px;
            margin-bottom: 20px;
            border-radius: 8px;
            font-weight: 600;
            display: flex;
            align-items: center;
            gap: 10px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            transition: all 0.3s ease;
        }}
        .status.loading {{
            background: #fff3cd;
            color: #856404;
            border-left: 4px solid #ffc107;
        }}
        .status.connected {{
            background: #d4edda;
            color: #155724;
            border-left: 4px solid #28a745;
        }}
        .status.error {{
            background: #f8d7da;
            color: #721c24;
            border-left: 4px solid #dc3545;
        }}
        .spinner {{
            width: 20px;
            height: 20px;
            border: 3px solid #f3f3f3;
            border-top: 3px solid #856404;
            border-radius: 50%;
            animation: spin 1s linear infinite;
        }}
        @keyframes spin {{
            0% {{ transform: rotate(0deg); }}
            100% {{ transform: rotate(360deg); }}
        }}
        .viewer-container {{
            background: white;
            border-radius: 10px;
            padding: 20px;
            box-shadow: 0 4px 6px rgba(0,0,0,0.1);
            margin-bottom: 20px;
        }}
        #dcv-display {{
            width: 100%;
            height: 800px;
            border: 2px solid #e0e0e0;
            border-radius: 8px;
            background: #000;
            /* 스크롤을 활성화하도록 overflow: hidden을 overflow: auto로 변경 */
            overflow: auto;
            position: relative;
        }}
        .info-panel {{
            background: white;
            padding: 20px;
            border-radius: 10px;
            box-shadow: 0 4px 6px rgba(0,0,0,0.1);
        }}
        .info-panel h3 {{
            color: #333;
            margin-bottom: 15px;
            font-size: 18px;
        }}
        .info-item {{
            display: flex;
            align-items: center;
            gap: 10px;
            padding: 10px;
            background: #f8f9fa;
            border-radius: 5px;
            margin-bottom: 10px;
        }}
        .info-item strong {{
            color: #667eea;
            min-width: 120px;
        }}
        .badge {{
            display: inline-block;
            padding: 4px 12px;
            border-radius: 12px;
            font-size: 12px;
            font-weight: 600;
            background: #667eea;
            color: white;
        }}
        .resolution-control {{
            display: flex;
            align-items: center;
            gap: 15px;
            padding: 10px 15px;
            background-color: #f9f9f9;
            border-radius: 8px;
            border: 1px solid #e1e1e1;
            margin-bottom: 15px;
        }}
        .resolution-label {{
            font-size: 14px;
            font-weight: 500;
            color: #333;
            white-space: nowrap;
        }}
        .resolution-options {{
            display: flex;
            gap: 8px;
            flex-wrap: wrap;
        }}
        .resolution-button {{
            padding: 6px 12px;
            border-radius: 4px;
            font-size: 13px;
            font-weight: 500;
            cursor: pointer;
            border: 1px solid #ccc;
            background: #fff;
            transition: all 0.2s ease;
        }}
        .resolution-button:hover {{
            border-color: #667eea;
        }}
        .resolution-button.active {{
            background: #667eea;
            color: white;
            border-color: #667eea;
        }}
        .resolution-info {{
            font-size: 12px;
            color: #666;
            margin-left: auto;
            white-space: nowrap;
        }}
        .control-panel {{
            display: flex;
            align-items: center;
            gap: 15px;
            padding: 10px 15px;
            background-color: #f9f9f9;
            border-radius: 8px;
            border: 1px solid #e1e1e1;
            margin-bottom: 15px;
        }}
        .control-label {{
            font-size: 14px;
            font-weight: 500;
            color: #333;
            white-space: nowrap;
        }}
        .control-button {{
            padding: 6px 16px;
            border-radius: 4px;
            font-size: 13px;
            font-weight: 500;
            cursor: pointer;
            border: 1px solid #ccc;
            transition: all 0.2s ease;
        }}
        .btn-take-control {{
            background: #28a745;
            color: white;
            border-color: #28a745;
        }}
        .btn-take-control:hover {{
            background: #218838;
            border-color: #1e7e34;
        }}
        .btn-give-control {{
            background: #dc3545;
            color: white;
            border-color: #dc3545;
        }}
        .btn-give-control:hover {{
            background: #c82333;
            border-color: #bd2130;
        }}
        .control-status {{
            font-size: 12px;
            color: #666;
            margin-left: auto;
        }}
    </style>
</head>
<body>
    <div class="container">
        <div class="header">
            <h1>🖥️ Browser Live View</h1>
            <div class="subtitle">
                Real-time streaming powered by AWS AgentCore & Amazon DCV
            </div>
        </div>

        <div id="status" class="status loading">
            <div class="spinner"></div>
            <span>Loading DCV SDK and connecting...</span>
        </div>

        <div class="viewer-container">
            <!-- 제어 Panel -->
            <div class="control-panel">
                <span class="control-label">Browser Control:</span>
                <button id="take-control-btn" class="control-button btn-take-control">
                    🎮 Human Take Over
                </button>
                <button id="give-control-btn" class="control-button btn-give-control" style="display:none">
                    🤖 Give Back Control
                </button>
                <span class="control-status" id="control-status">Automation Active</span>
            </div>
            
            <!-- 해상도 Control -->
            <div class="resolution-control">
                <span class="resolution-label">Display Size:</span>
                <div class="resolution-options" id="resolution-buttons">
                    <button class="resolution-button" data-resolution="720p" title="1280×720">720p</button>
                    <button class="resolution-button active" data-resolution="900p" title="1600×900">900p</button>
                    <button class="resolution-button" data-resolution="1080p" title="1920×1080">1080p</button>
                    <button class="resolution-button" data-resolution="1440p" title="2560×1440">1440p</button>
                </div>
                <span class="resolution-info" id="current-resolution">Current: 1600×900</span>
            </div>

            <div id="dcv-display"></div>
        </div>

        <div class="info-panel">
            <h3>ℹ️ Session Information</h3>
            <div class="info-item">
                <strong>Session ID:</strong>
                <span>{session_id}</span>
            </div>
            <div class="info-item">
                <strong>Streaming Protocol:</strong>
                <span>Amazon DCV (NICE DCV)</span>
            </div>
            <div class="info-item">
                <strong>Browser Backend:</strong>
                <span>AWS Bedrock AgentCore</span>
            </div>
            <div class="info-item">
                <strong>Status:</strong>
                <span id="connection-status" class="badge">Connecting...</span>
            </div>
        </div>
    </div>

    <script>
        // 자동 삽입된 구성
        const CONFIG = {{
            presignedUrl: '{presigned_url}',
            sessionId: '{session_id}',
            authToken: '{session_id}',
            controlApiUrl: '{CONTROL_API_URL}'  // 삽입된 Control API URL
        }};

        // 해상도 설정
        const RESOLUTIONS = {{
            '720p': {{ width: 1280, height: 720 }},
            '900p': {{ width: 1600, height: 900 }},
            '1080p': {{ width: 1920, height: 1080 }},
            '1440p': {{ width: 2560, height: 1440 }}
        }};
        let currentResolution = '900p'; // 기본 해상도
        let hasControl = false; // 제어 상태 추적

        const statusDiv = document.getElementById('status');
        const statusBadge = document.getElementById('connection-status');
        const spinnerDiv = statusDiv.querySelector('.spinner');
        const resolutionInfo = document.getElementById('current-resolution');
        const resolutionButtons = document.getElementById('resolution-buttons').querySelectorAll('.resolution-button');
        const takeControlBtn = document.getElementById('take-control-btn');
        const giveControlBtn = document.getElementById('give-control-btn');
        const controlStatus = document.getElementById('control-status');
        let dcvConnection = null;

        // 해상도 버튼 초기화
        resolutionButtons.forEach(button => {{
            button.addEventListener('click', () => {{
                const resolution = button.dataset.resolution;
                changeResolution(resolution);
            }});
        }});

        // Control 버튼 초기화
        takeControlBtn.addEventListener('click', takeControl);
        giveControlBtn.addEventListener('click', giveBackControl);

        // 제어권 인수 function
        async function takeControl() {{
            try {{
                updateStatus('Taking control of browser...', 'loading', 'Taking Control');
                
                const response = await fetch(`${{CONFIG.controlApiUrl}}/api/update_browser_stream`, {{
                    method: 'POST',
                    headers: {{
                        'Content-Type': 'application/json'
                    }},
                    body: JSON.stringify({{
                        automationStreamUpdate: {{
                            streamStatus: 'disabled'
                        }}
                    }})
                }});
                
                if (response.ok) {{
                    const result = await response.json();
                    hasControl = true;
                    takeControlBtn.style.display = 'none';
                    giveControlBtn.style.display = 'inline-block';
                    controlStatus.textContent = '🎮 Human Control Active';
                    updateStatus('✅ You now have control of the browser', 'connected', 'Human Control');
                    console.log('Successfully took control of browser:', result);
                }} else {{
                    const errorText = await response.text();
                    updateStatus('❌ Failed to take control of browser', 'error', 'Error');
                    console.error('Failed to take control:', errorText);
                }}
            }} catch (error) {{
                console.error('Error taking control:', error);
                updateStatus(`❌ Error: ${{error.message}}`, 'error', 'Error');
            }}
        }}

        // 제어권 반환 function
        async function giveBackControl() {{
            try {{
                updateStatus('Giving back control...', 'loading', 'Releasing Control');
                
                const response = await fetch(`${{CONFIG.controlApiUrl}}/api/update_browser_stream`, {{
                    method: 'POST',
                    headers: {{
                        'Content-Type': 'application/json'
                    }},
                    body: JSON.stringify({{
                        automationStreamUpdate: {{
                            streamStatus: 'enabled'
                        }}
                    }})
                }});
                
                if (response.ok) {{
                    const result = await response.json();
                    hasControl = false;
                    takeControlBtn.style.display = 'inline-block';
                    giveControlBtn.style.display = 'none';
                    controlStatus.textContent = 'Automation Active';
                    updateStatus('✅ Returned control to automation', 'connected', 'Automation Active');
                    console.log('Successfully gave back control:', result);
                }} else {{
                    const errorText = await response.text();
                    updateStatus('❌ Failed to give back control', 'error', 'Error');
                    console.error('Failed to give back control:', errorText);
                }}
            }} catch (error) {{
                console.error('Error giving back control:', error);
                updateStatus(`❌ Error: ${{error.message}}`, 'error', 'Error');
            }}
        }}

        function changeResolution(resolutionId) {{
            // 먼저 UI update
            currentResolution = resolutionId;

            // 버튼 update
            resolutionButtons.forEach(btn => {{
                btn.classList.toggle('active', btn.dataset.resolution === resolutionId);
            }});

            // 정보 text update
            const resolution = RESOLUTIONS[resolutionId];
            resolutionInfo.textContent = `Current: ${{resolution.width}}×${{resolution.height}}`;

            // 연결되었으면 DCV display layout update
            if (dcvConnection && dcvConnection.requestDisplayLayout) {{
                console.log(`Changing resolution to: ${{resolution.width}}×${{resolution.height}}`);

                dcvConnection.requestDisplayLayout([{{
                    name: "Main Display",
                    rect: {{
                        x: 0,
                        y: 0,
                        width: resolution.width,
                        height: resolution.height
                    }},
                    primary: true
                }}]);

                // 더 잘 보이도록 container 화면 비율 조정
                const container = document.getElementById('dcv-display');
                if (container) {{
                    const aspectRatio = resolution.height / resolution.width;
                    const containerWidth = container.clientWidth;
                    const newHeight = Math.min(containerWidth * aspectRatio, 800);
                    container.style.height = `${{newHeight}}px`;
                }}
            }}
        }}

        function updateStatus(message, type = 'loading', badgeText = null) {{
            const span = statusDiv.querySelector('span');
            span.textContent = message;
            statusDiv.className = `status ${{type}}`;

            if (type === 'loading') {{
                spinnerDiv.style.display = 'block';
            }} else {{
                spinnerDiv.style.display = 'none';
            }}

            if (badgeText && statusBadge) {{
                statusBadge.textContent = badgeText;
                statusBadge.style.background = type === 'connected' ? '#28a745' :
                                              type === 'error' ? '#dc3545' : '#ffc107';
            }}
        }}

        function loadDCVSDK() {{
            return new Promise((resolve, reject) => {{
                updateStatus('📦 Loading DCV SDK...', 'loading', 'Loading SDK');

                const script = document.createElement('script');
                script.src = '/dcv-sdk/dcvjs-umd/dcv.js';

                script.onload = () => {{
                    if (!window.dcv) {{
                        reject(new Error('DCV SDK loaded but window.dcv not available'));
                        return;
                    }}

                    // HOTFIX: 호출 전에 setWorkerPath가 있는지 확인
                    // 일부 DCV SDK 버전에는 이 function이 없을 수 있음
                    if (typeof window.dcv.setWorkerPath === 'function') {{
                        try {{
                            window.dcv.setWorkerPath(
                                window.location.origin + '/dcv-sdk/dcvjs-umd/dcv/'
                            );
                            console.log('✅ Worker path configured');
                        }} catch (e) {{
                            console.warn('⚠️ Could not set worker path:', e);
                            // connect()의 baseUrl이 처리하므로 계속 진행
                        }}
                    }} else {{
                        console.log('ℹ️ setWorkerPath not available, will use baseUrl in connect()');
                    }}

                    console.log('✅ DCV SDK loaded successfully');
                    resolve();
                }};

                script.onerror = () => {{
                    reject(new Error('Failed to load DCV SDK. Check if dcv-sdk files exist.'));
                }};

                document.head.appendChild(script);
            }});
        }}

        function authenticateWithDCV() {{
            return new Promise((resolve, reject) => {{
                updateStatus('🔐 Authenticating with DCV server...', 'loading', 'Authenticating');

                window.dcv.authenticate(CONFIG.presignedUrl, {{
                    promptCredentials: (authType, callback) => {{
                        // Presigned URL에는 자격 증명이 URL에 포함됨
                        // 사용자에게 자격 증명을 요청할 필요 없음
                        console.log('📝 Credentials prompt (using presigned URL)');
                        callback(null, null);
                    }},
                    httpExtraSearchParams: (method, url, body) => {{
                        // Presigned URL에서 query parameter를 추출해 WebSocket에 전달
                        try {{
                            const parsedUrl = new URL(CONFIG.presignedUrl);
                            const searchParams = parsedUrl.searchParams;
                            console.log('📡 Adding auth params to WebSocket request');
                            return searchParams;
                        }} catch (e) {{
                            console.error('Failed to parse presigned URL:', e);
                            return new URLSearchParams();
                        }}
                    }},
                    success: (auth, result) => {{
                        console.log('✅ Authentication successful');
                        const credentials = result[0];
                        resolve({{
                            presignedUrl: CONFIG.presignedUrl,
                            sessionId: credentials.sessionId,
                            authToken: credentials.authToken
                        }});
                    }},
                    error: (auth, error) => {{
                        console.error('❌ Authentication failed:', error);
                        reject(new Error('Authentication failed: ' + JSON.stringify(error)));
                    }}
                }});
            }});
        }}

        function connectToDCV(credentials) {{
            updateStatus('🔌 Connecting to browser session...', 'loading', 'Connecting');

            const resolution = RESOLUTIONS[currentResolution];
            console.log(`Initial resolution: ${{resolution.width}}×${{resolution.height}}`);

            const dcvConfig = {{
                url: credentials.presignedUrl,
                sessionId: credentials.sessionId,
                authToken: credentials.authToken,
                divId: 'dcv-display',
                baseUrl: window.location.origin + '/dcv-sdk/dcvjs-umd',
                observers: {{
                    httpExtraSearchParams: (method, url, body) => {{
                        // Streaming 연결용 query parameter를 presigned URL에서 추출
                        try {{
                            const parsedUrl = new URL(credentials.presignedUrl);
                            const searchParams = parsedUrl.searchParams;
                            console.log('📡 Adding auth params to streaming connection');
                            return searchParams;
                        }} catch (e) {{
                            console.error('Failed to parse presigned URL:', e);
                            return new URLSearchParams();
                        }}
                    }},
                    firstFrame: () => {{
                        console.log('✅ First frame received');
                        updateStatus('✅ Connected! Streaming browser view...', 'connected', 'Live');

                        // 연결된 후 초기 해상도 적용
                        setTimeout(() => {{
                            if (dcvConnection && dcvConnection.requestDisplayLayout) {{
                                dcvConnection.requestDisplayLayout([{{
                                    name: "Main Display",
                                    rect: {{
                                        x: 0,
                                        y: 0,
                                        width: resolution.width,
                                        height: resolution.height
                                    }},
                                    primary: true
                                }}]);
                                console.log(`Applied initial resolution: ${{resolution.width}}×${{resolution.height}}`);
                            }}
                        }}, 1000);
                    }},
                    displayLayout: (serverWidth, serverHeight, heads) => {{
                        console.log(`Display layout changed: ${{serverWidth}}×${{serverHeight}}`);
                    }},
                    error: (error) => {{
                        console.error('❌ Connection error:', error);
                        updateStatus('❌ Connection error: ' + JSON.stringify(error), 'error', 'Error');
                    }},
                    close: (closeInfo) => {{
                        console.log('Connection closed:', closeInfo);
                        updateStatus('⚠️ Connection closed', 'error', 'Disconnected');
                    }}
                }}
            }};

            window.dcv.connect(dcvConfig)
                .then(connection => {{
                    console.log('Connection established');
                    dcvConnection = connection;

                    // 초기 해상도 적용
                    setTimeout(() => {{
                        if (connection.requestDisplayLayout) {{
                            connection.requestDisplayLayout([{{
                                name: "Main Display",
                                rect: {{
                                    x: 0,
                                    y: 0,
                                    width: resolution.width,
                                    height: resolution.height
                                }},
                                primary: true
                            }}]);
                            console.log(`Applied initial resolution: ${{resolution.width}}×${{resolution.height}}`);
                        }}
                    }}, 500);
                }})
                .catch(error => {{
                    console.error('Connection failed:', error);
                    updateStatus('❌ Connection error: ' + JSON.stringify(error), 'error', 'Error');
                }});
        }}

        async function initialize() {{
            try {{
                console.log('🚀 Starting live view initialization...');
                console.log('Session ID:', CONFIG.sessionId);

                await loadDCVSDK();
                const credentials = await authenticateWithDCV();
                connectToDCV(credentials);

            }} catch (error) {{
                console.error('Initialization failed:', error);
                updateStatus('❌ Error: ' + error.message, 'error', 'Failed');
            }}
        }}

        window.addEventListener('load', initialize);

        window.addEventListener('beforeunload', () => {{
            if (window.dcv && typeof window.dcv.disconnect === 'function') {{
                window.dcv.disconnect();
            }}
        }});
    </script>
</body>
</html>"""

    with open(HTML_FILE, "w", encoding="utf-8") as f:
        f.write(html_content)

    print(f"   ✅ Created {HTML_FILE}")
    print("   📍 Presigned URL injected (expires in 5 minutes)")
    print(f"   🔌 Control API URL configured: {CONTROL_API_URL}")
    return True


def verify_file_structure():
    """필요한 파일이 모두 있는지 확인합니다."""
    print("\n🔍 Verifying file structure...")

    required = [
        (HTML_FILE, "HTML page"),
        ("dcv-sdk/dcvjs-umd/dcv.js", "DCV SDK (UMD)"),
        ("dcv-sdk/dcvjs-umd/dcv", "DCV workers directory"),
    ]

    all_good = True
    for path, description in required:
        exists = os.path.exists(path)
        icon = "✓" if exists else "✗"
        status = "Found" if exists else "MISSING"
        print(f"   {icon} {description}: {status}")
        if not exists:
            all_good = False

    return all_good


def start_web_server():
    """백그라운드 스레드에서 HTTP 서버를 시작합니다."""

    class QuietHandler(http.server.SimpleHTTPRequestHandler):
        def log_message(self, format, *args):
            # 오류만 log로 기록
            if args[1] != "200":
                super().log_message(format, *args)

    handler = QuietHandler

    try:
        httpd = socketserver.TCPServer(("", SERVER_PORT), handler)

        # Background thread에서 server 시작
        server_thread = threading.Thread(target=httpd.serve_forever, daemon=True)
        server_thread.start()

        return httpd, server_thread
    except OSError as e:
        if e.errno == 48 or e.errno == 98:  # Address가 이미 사용 중
            print(f"   ⚠️  Port {SERVER_PORT} already in use")
            print("   ℹ️  Server might already be running")
            return None, None
        raise


# ============================================================================
# Main 실행
# ============================================================================

print("=" * 70)
print("🚀 AUTOMATIC LIVE VIEW SETUP")
print("=" * 70)

try:
    # 1단계: Presigned URL이 포함된 HTML 생성
    if not create_html_with_url(live_view_url, session_data["sessionId"]):
        raise Exception("Failed to create HTML file")

    # 2단계: File 구조 검증
    if not verify_file_structure():
        print("\n❌ Missing required files!")
        print("   Please make sure you ran the DCV SDK download step first.")
        print("\n⚠️  Skipping web server setup due to missing files.")
    else:
        print("\n✅ All files ready!")

        # 3단계: Web server 시작
        print("\n🌐 Starting web server...")
        httpd, server_thread = start_web_server()

        if httpd:
            print(f"   ✅ Server running on port {SERVER_PORT}")
            print(f"   📍 URL: {SERVER_URL}")

            # 4단계: 브라우저 열기
            print("\n🌐 Opening browser...")
            time.sleep(1)  # Server 시작 시간 확보

            try:
                webbrowser.open(SERVER_URL)
                print("   ✅ Browser opened!")
            except Exception as e:
                print(f"   ⚠️  Could not auto-open browser: {e}")
                print(f"   ℹ️  Please manually open: {SERVER_URL}")

            # 성공 메시지
            print("\n" + "=" * 70)
            print("✅ LIVE VIEW IS READY!")
            print("=" * 70)
            print("\n📺 What you should see in the browser:")
            print("   1. Status: Loading... → Authenticating... → Connected!")
            print("   2. Live browser stream appears in the black area")
            print("   3. Human Take Over button to control the browser")
            print("   4. Resolution controls to adjust the display size")
            print("   5. Session information displayed below")
            print("   6. You can scroll within the browser view using mouse wheel")
            print("\n⚠️  IMPORTANT:")
            print("   - Server is running in background")
            print("   - Presigned URL expires in 5 minutes")
            print("   - Control API running on port 8081")
            print("   - To stop server: Click ■ (stop) button or restart kernel")
            print("\n🔗 Access URL: " + SERVER_URL)
            print("=" * 70)

            # Server 정보에 계속 접근할 수 있도록 유지
            print("\n💡 Tip: Leave this cell running to keep the server active.")

        else:
            print("\n⚠️  Server might already be running")
            print(f"   Try opening: {SERVER_URL}")

except Exception as e:
    print(f"\n❌ Setup failed: {e}")
    import traceback

    traceback.print_exc()
    print("\n💡 Troubleshooting:")
    print("   1. Make sure you ran Step 0 and Step 1 first")
    print("   2. Verify DCV SDK is downloaded (check previous cells)")
    print("   3. Check if port 8080 is available")

## 정리: 브라우저 세션 중지

테스트가 끝나면 불필요한 비용이 발생하지 않도록 브라우저 세션을 중지하세요.

In [ ]:
# # 브라우저 세션 중지
if browser_client:
    browser_client.stop()
    print("✅ Browser session stopped")
else:
    print("⚠️ No active browser session to stop")

## 문제 해결

### 일반적인 문제

#### 1. "Failed to load DCV SDK"

**문제:** HTML 페이지에서 DCV SDK file을 찾을 수 없습니다.

**해결 방법:**
- 프로젝트에 `dcv-sdk/dcvjs-umd/dcv.js`가 있는지 확인합니다.
- HTML을 직접 열지 않고 HTTP를 통해 file을 제공하고 있는지 확인합니다.
- `python -m http.server 8080`을 사용해 file을 제공합니다.

#### 2. "Authentication failed"

**문제:** Presigned URL이 유효하지 않거나 만료되었습니다.

**해결 방법:**
- Presigned URL은 기본적으로 5분 후 만료됩니다.
- 2.1단계를 다시 실행해 새 URL을 생성하고 3단계에서 새 URL을 web app에 삽입합니다. 로컬 브라우저 페이지를 새로 고치면 기존 AgentCore Browser 세션에 다시 연결됩니다.
- 매우 긴 URL 전체를 복사했는지 확인합니다.

#### 3. 검은 화면 / Video 없음

**문제:** DCV가 연결되지만 브라우저가 표시되지 않습니다.

**해결 방법:**
- 브라우저 console에서 오류를 확인합니다.
- `setWorkerPath`가 올바른지 확인합니다.
- `dcv.connect()`에 `baseUrl` parameter가 설정되어 있는지 확인합니다.

#### 4. CORS 오류

**문제:** 브라우저가 CORS policy로 인해 request를 차단합니다.

**해결 방법:**
- HTML file을 직접 열지 마세요(file:// protocol).
- 로컬 web server를 사용합니다: `python -m http.server 8080`
- Backend API를 사용한다면 Flask CORS가 구성되어 있는지 확인합니다.

### 도움말

문제가 계속되면 다음을 확인하세요.

1. 브라우저 console(F12)에서 오류 메시지 확인
2. 모든 file path가 올바른지 확인
3. AWS 자격 증명이 구성되어 있는지 확인
4. 사용 중인 리전에서 AgentCore Browser 서비스를 사용할 수 있는지 확인

## 전체 구현 예제

React, WebSocket update 및 production 기능이 포함된 전체 구현 예제는 다음을 참고하세요.

**GitHub Repository:**
https://github.com/aws-samples/sample-browser-order-automation-agentcore

이 sample에는 다음이 포함됩니다.
- 전체 React 애플리케이션
- FastAPI backend
- 실시간 상태 update
- 세션 녹화
- Multi-agent orchestration

### 살펴볼 주요 File

- `frontend/src/components/LiveBrowserViewer.js` - React component implementation
- `backend/services/browser_service.py` - Browser session management
- `backend/app.py` - API endpoints

## 요약

이 튜토리얼에서는 다음 내용을 알아보았습니다.

✅ Python에서 AgentCore Browser client 설정

✅ 안전한 스트리밍을 위한 presigned URL 생성

✅ DCVjs SDK 다운로드 및 구성

✅ 브라우저 화면을 실시간으로 스트리밍하는 HTML 페이지 생성

✅ 브라우저를 수동 제어하는 Human Take Over 버튼 추가

✅ 화면 크기를 조정하는 해상도 control 추가(720p, 900p, 1080p, 1440p)

✅ 인증과 연결을 올바르게 처리

✅ 자동화 모드 전환용 control API 구현

✅ 일반적인 문제 해결

### 다음 단계

- HTML 페이지에 더 많은 대화형 control 추가
- AI Agent(Nova Act, Strands)와 통합
- FastAPI로 production backend 구축
- 세션 녹화 구현
- 사용자 인증 추가

**즐겁게 만들어 보세요! 🚀**